# NB6 — Encodeur transformer figé

Notebook du pipeline **P24**.

## Portée du notebook

Ce notebook utilise un **encodeur transformer figé** pour produire des représentations contextuelles,
puis entraîne une petite tête MLP au-dessus.

On conserve le `test.csv` déjà fourni et on construit uniquement un **jeu de validation stratifié à partir du train**.

In [ ]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn tensorflow transformers openpyxl

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from transformers import AutoTokenizer, TFAutoModel

from nlp_disaster_utils import (
    seed_everything,
    load_train_test_xy,
    stratified_validation_split,
    evaluate_probability_outputs,
    round_results,
    metric_matrix_from_results,
    save_results_bundle,
)

seed_everything(42)
print("TensorFlow :", tf.__version__)

In [ ]:
DATA_DIR = "../../data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False

VAL_SIZE_WITHIN_TRAIN = 0.10
RANDOM_STATE = 42

MODEL_NAME = "distilbert-base-uncased"
MAX_LEN = 96
BATCH_SIZE = 16
EPOCHS = 4
LEARNING_RATE = 1e-3

OUTPUT_STEM = "NB6_frozen_transformer_encoder"
RESULTS_DIR = "results"

In [ ]:
df_train, X_train_full, y_train_full, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

X_train, X_val, y_train, y_val = stratified_validation_split(
    X_train_full,
    y_train_full,
    val_size=VAL_SIZE_WITHIN_TRAIN,
    random_state=RANDOM_STATE,
)

print("Taille train :", len(X_train))
print("Taille validation :", len(X_val))
print("Taille test :", len(X_test))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
encoder = TFAutoModel.from_pretrained(MODEL_NAME)
encoder.trainable = False

def tokenize_texts(texts):
    return tokenizer(
        list(texts),
        padding="max_length",
        truncation=True,
        max_length=MAX_LEN,
        return_tensors="tf",
    )

train_tokens = tokenize_texts(X_train)
val_tokens = tokenize_texts(X_val)
test_tokens = tokenize_texts(X_test)

In [ ]:
def build_frozen_transformer_mlp():
    input_ids = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="input_ids")
    attention_mask = keras.Input(shape=(MAX_LEN,), dtype=tf.int32, name="attention_mask")

    outputs = encoder(input_ids=input_ids, attention_mask=attention_mask)
    x = outputs.last_hidden_state[:, 0, :]   # Représentation de type CLS pour DistilBERT
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inputs=[input_ids, attention_mask], outputs=out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.BinaryAccuracy(name="accuracy"),
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="roc_auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
        ],
    )
    return model

model = build_frozen_transformer_mlp()
model.summary()

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=1, restore_best_weights=True)
]

history = model.fit(
    x={"input_ids": train_tokens["input_ids"], "attention_mask": train_tokens["attention_mask"]},
    y=np.array(y_train),
    validation_data=(
        {"input_ids": val_tokens["input_ids"], "attention_mask": val_tokens["attention_mask"]},
        np.array(y_val),
    ),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=callbacks,
    verbose=1,
)

train_scores = model.predict(
    {"input_ids": train_tokens["input_ids"], "attention_mask": train_tokens["attention_mask"]},
    batch_size=BATCH_SIZE,
    verbose=0,
).ravel()

test_scores = model.predict(
    {"input_ids": test_tokens["input_ids"], "attention_mask": test_tokens["attention_mask"]},
    batch_size=BATCH_SIZE,
    verbose=0,
).ravel()

results_df = pd.DataFrame([
    evaluate_probability_outputs(
        name="P24_FrozenDistilBERT_MLP",
        y_train=np.array(y_train),
        train_scores=train_scores,
        y_test=np.array(y_test),
        test_scores=test_scores,
        threshold=0.5,
    )
])

results_df = round_results(results_df)
results_df

In [ ]:
display(results_df)
metric_view = round_results(metric_matrix_from_results(results_df))
display(metric_view)

save_results_bundle(results_df, output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX enregistrés dans ./{RESULTS_DIR}")